## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             log_loss, average_precision_score, mean_squared_error, mean_absolute_error, r2_score)
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

## Setup

In [2]:
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_V4_0", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


## Dataloader

In [4]:
def prep_split(tr_idx, te_idx):
    """Standardise using the training rows only (no missing values left in the regression set)."""
    sc = StandardScaler().fit(X_full[tr_idx])
    return sc.transform(X_full[tr_idx]), sc.transform(X_full[te_idx]), y_raw[tr_idx], y_raw[te_idx]

# same 80/20 split as every earlier version (seed 42)
idx_all = np.arange(len(y_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
Xtr, Xte, ytr, yte = prep_split(tr_idx, te_idx)
print(f"train {Xtr.shape} test {Xte.shape}")

train (1769, 13) test (443, 13)


## Model

In [5]:
EPOCHS_MAX, BATCH, LR, WD = 100, 64, 1e-3, 1e-4     # same optimiser settings as V2_0
device = torch.device("cpu")

def run_epochs(model, X, y, loss_fn, epochs, seed, on_epoch=None):
    """Adam + cosine (T_max = 100), mini-batch 64. on_epoch(epoch, model) is called after every epoch."""
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_MAX)
    Xt = torch.from_numpy(np.asarray(X, dtype=np.float32)); yt = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad(); loss_fn(model(Xt[idx]), yt[idx]).backward(); opt.step()
        sch.step()
        if on_epoch is not None:
            on_epoch(ep, model)
    return model

def smooth(a, w=5):
    return np.convolve(np.pad(a, (w // 2, w // 2), mode="edge"), np.ones(w) / w, mode="valid")

def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))

In [6]:
class SubNet(nn.Module):
    """One feature -> one number. ffn='relu': Linear(1,hid)-ReLU-Linear(hid,1).
       ffn='reglu': Linear(1,2*hid) split into (value, gate) -> value * ReLU(gate) -> Linear(hid,1)."""
    def __init__(self, ffn="relu", hid=64):
        super().__init__()
        self.ffn = ffn
        self.fc1 = nn.Linear(1, hid if ffn == "relu" else 2 * hid); self.fc2 = nn.Linear(hid, 1); self.relu = nn.ReLU()
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu"); nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity="relu"); nn.init.zeros_(self.fc2.bias)
    def forward(self, xj):
        z = self.fc1(xj)
        if self.ffn == "reglu":
            val, gate = z.chunk(2, dim=-1); a = val * self.relu(gate)
        else:
            a = self.relu(z)
        return self.fc2(a)

class NAM(nn.Module):
    def __init__(self, n_feat=13, ffn="relu", hid=64):
        super().__init__()
        self.subs = nn.ModuleList([SubNet(ffn, hid) for _ in range(n_feat)])
        self.b0 = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        return sum(sub(x[:, j:j + 1]) for j, sub in enumerate(self.subs)) + self.b0

# ReLU baseline: hidden 64 (193 parameters per subnet). ReGLU "same hidden": 64 (321 per subnet).
# ReGLU "param-matched": hidden 38 (191 per subnet, closest to 193).
VARIANTS = {
    "ReLU (baseline)":       dict(ffn="relu", hid=64),
    "ReGLU, same hidden":    dict(ffn="reglu", hid=64),
    "ReGLU, param-matched":  dict(ffn="reglu", hid=38),
}
DELTA = 0.5   # Huber delta (log-z space), as V2_0
def make_model(v, seed):
    torch.manual_seed(seed)
    return NAM(n_feat=len(REG_FEATURES), **VARIANTS[v]).to(device)

## Sanity Check

In [7]:
ref = n_params(make_model("ReLU (baseline)", 0))
for v in VARIANTS:
    print(f"{v:24s} parameters: {n_params(make_model(v, 0)):6d}")
assert abs(n_params(make_model("ReGLU, param-matched", 0)) - ref) / ref < 0.02, "param-matched variant is more than 2% off"
xb = torch.randn(8, len(REG_FEATURES)); yb = torch.randn(8, 1); lf = nn.HuberLoss(delta=DELTA)
for v in VARIANTS:
    m = make_model(v, 0); m.train()
    loss = lf(m(xb), yb); loss.backward()
    assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None)
print("gradient check OK: loss and all gradients finite for every variant")
sub = SubNet("reglu", 38).eval()
x1 = torch.randn(6, 1)
with torch.no_grad():
    z = sub.fc1(x1); val, gate = z[:, :38], z[:, 38:]
    assert torch.allclose(sub.fc2(val * torch.clamp(gate, min=0)), sub(x1), atol=1e-6)
print("ReGLU subnet matches value * ReLU(gate)")

ReLU (baseline)          parameters:   2510
ReGLU, same hidden       parameters:   4174
ReGLU, param-matched     parameters:   2484
gradient check OK: loss and all gradients finite for every variant
ReGLU subnet matches value * ReLU(gate)


## Training

In [8]:
INNER_K = 3
def to_z(y_raw_, scaler): return scaler.transform(np.log1p(y_raw_).reshape(-1, 1)).ravel()
def to_dollar(z, scaler): return np.expm1(np.clip(scaler.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel(), 0.0, 15.0))   # clip keeps a bad early epoch finite

def predict_dollar(model, X, scaler):
    model.eval()
    with torch.no_grad():
        return to_dollar(model(torch.from_numpy(np.asarray(X, dtype=np.float32))).numpy().ravel(), scaler)

def fit_eval(variant, Xa, ya, Xb, yb, seed):
    """Epoch count comes from an inner K-fold on (Xa, ya) only; (Xb, yb) is scored once."""
    loss_fn = nn.HuberLoss(delta=DELTA)
    curves = []
    for k, (a, b) in enumerate(KFold(INNER_K, shuffle=True, random_state=seed).split(Xa)):
        sc = StandardScaler().fit(np.log1p(ya[a]).reshape(-1, 1))
        rm = []
        run_epochs(make_model(variant, seed + k), Xa[a], to_z(ya[a], sc), loss_fn, EPOCHS_MAX, seed + k,
                   on_epoch=lambda ep, m, b=b, sc=sc: rm.append(float(np.sqrt(mean_squared_error(ya[b], predict_dollar(m, Xa[b], sc))))))
        curves.append(rm)
    E = int(np.argmin(smooth(np.mean(curves, axis=0)))) + 1   # epoch chosen on inner validation only
    sc = StandardScaler().fit(np.log1p(ya).reshape(-1, 1))
    model = run_epochs(make_model(variant, seed + 99), Xa, to_z(ya, sc), loss_fn, E, seed + 99)
    pred = predict_dollar(model, Xb, sc)
    mse = mean_squared_error(yb, pred)
    row = {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse,
           "epochs": E, "n_params": n_params(model)}
    return row, pred

In [9]:
results, scores = [], {}
for v in VARIANTS:
    t0 = time.time()
    row, s = fit_eval(v, Xtr, ytr, Xte, yte, SEED)
    scores[v] = s
    results.append(dict(model=f"NAM (log1p + Huber) | {v}", tier="extracurricular", selection="inner validation",
                        **{k: round(float(x), 4) for k, x in row.items()}))
    print(f"{v:24s} R2 = {row['r2']:.4f} | epochs {row['epochs']} | {time.time() - t0:.0f}s")

ReLU (baseline)          R2 = 0.7194 | epochs 95 | 41s


ReGLU, same hidden       R2 = 0.7621 | epochs 97 | 47s


ReGLU, param-matched     R2 = 0.7449 | epochs 97 | 53s


## Evaluation

In [10]:
table = pd.DataFrame(results)
print(table.drop(columns=["tier", "selection"]).to_string(index=False))
json.dump({"n_test": int(len(yte)), "seed": SEED, "models": results}, open("weights_V4_0/regression_pytorch_test_metrics.json", "w"), indent=2)
np.savez("weights_V4_0/regression_pytorch_test_scores.npz", y_true=yte, **scores)

                                     model     r2       rmse       mae          mse  epochs  n_params
     NAM (log1p + Huber) | ReLU (baseline) 0.7194 11544.4099 7211.3900 1.332734e+08    95.0    2510.0
  NAM (log1p + Huber) | ReGLU, same hidden 0.7621 10630.7805 6894.9313 1.130135e+08    97.0    4174.0
NAM (log1p + Huber) | ReGLU, param-matched 0.7449 11008.5804 6986.1992 1.211888e+08    97.0    2484.0
